# Notebook 02: Alkyne VQE Simulation
## Quantum Simulation of Alkynes via PySCF + OpenFermion + PennyLane

**Molecules:** Acetylene (C₂H₂)  
**Key physics:** Triple C≡C bond, cylindrical π-system, stronger electron correlation than alkenes  
**Methods:** HF → CCSD → VQE (UCCSD ansatz, Adam optimizer)  
**Target:** Chemical accuracy (|ΔE| < 1.6 mHa vs FCI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/quantum-alkene-alkyne-pyscf/blob/main/notebooks/02_alkyne_vqe_simulation.ipynb)

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────
import sys, subprocess
pkgs = ['pyscf>=2.5','openfermion>=1.6','openfermionpyscf>=0.5',
        'pennylane>=0.38','tqdm','seaborn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Dependencies installed.')

In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
from tqdm.auto import trange
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

from pyscf import gto, scf, cc
from openfermion.chem import MolecularData
from openfermion import get_fermion_operator, jordan_wigner, bravyi_kitaev
from openfermion.utils import count_qubits
try:
    from openfermion.transforms import freeze_orbitals
except ImportError:
    from openfermion.utils import freeze_orbitals
from openfermionpyscf import run_pyscf
import pennylane as qml
from pennylane import qchem
print(f'PennyLane {qml.__version__} | All imports OK.')

In [ ]:
# ── CELL 3: PennyLane compatibility shim ─────────────────────────────
_PL_NEW = tuple(int(x) for x in qml.__version__.split('.')[:2]) >= (0, 38)

def _pauli_gate(name, wire):
    return {'X': qml.PauliX, 'Y': qml.PauliY, 'Z': qml.PauliZ}[name](wire)

def openfermion_to_pennylane(qubit_op):
    coeffs, ops = [], []
    for term, coeff in qubit_op.terms.items():
        if abs(np.real(coeff)) < 1e-15:
            continue
        coeffs.append(np.real(coeff))
        if not term:
            ops.append(qml.Identity(0))
        else:
            gates = [_pauli_gate(p, i) for i, p in term]
            ops.append(gates[0] if len(gates) == 1 else qml.prod(*gates))
    if _PL_NEW:
        return qml.ops.LinearCombination(np.array(coeffs), ops)
    return qml.Hamiltonian(np.array(coeffs), ops)

print('Hamiltonian builder ready.')

In [ ]:
# ── CELL 4: Acetylene geometry + classical references ─────────────────
# Linear D∞h molecule, bond lengths in Angstroms
# C≡C = 1.203 Å, C-H = 1.063 Å (experimental)
acetylene_geometry = [
    ('C', (0.000, 0.000,  0.000)),
    ('C', (0.000, 0.000,  1.203)),
    ('H', (0.000, 0.000, -1.063)),
    ('H', (0.000, 0.000,  2.266)),
]

BASIS = 'sto-3g'
acc = MolecularData(geometry=acetylene_geometry, basis=BASIS,
                    multiplicity=1, charge=0, description='acetylene')
acc = run_pyscf(acc, run_scf=True, run_ccsd=True, run_fci=True)

print('Acetylene classical reference energies (STO-3G):')
print(f'  HF   : {acc.hf_energy:.8f} Ha')
print(f'  CCSD : {acc.ccsd_energy:.8f} Ha')
print(f'  FCI  : {acc.fci_energy:.8f} Ha')
print(f'  Correlation energy (FCI-HF): {(acc.fci_energy - acc.hf_energy)*1000:.3f} mHa')
print(f'  n_electrons : {acc.n_electrons}')
print(f'  n_orbitals  : {acc.n_orbitals}')
print(f'  n_qubits (JW, full): {acc.n_qubits}')

In [ ]:
# ── CELL 5: Qubit Hamiltonian + active space ──────────────────────────
# Freeze 1 core sigma orbital (2 core electrons)
# Active space: 8 electrons, 4 orbitals → 8 qubits (JW)
fermion_ham = get_fermion_operator(acc.get_molecular_hamiltonian())

jw_full = jordan_wigner(fermion_ham)
n_q_full = count_qubits(jw_full)

active_fham = freeze_orbitals(fermion_ham, occupied=[0], virtual=[])
jw_active = jordan_wigner(active_fham)
n_q_active = count_qubits(jw_active)
n_elec_active = acc.n_electrons - 2

print(f'Full JW: {n_q_full} qubits')
print(f'Active JW: {n_q_active} qubits | Active electrons: {n_elec_active}')
print(f'Pauli terms (active JW): {len(list(jw_active.terms))}')

In [ ]:
# ── CELL 6: Build PennyLane circuit components ────────────────────────
H_pl = openfermion_to_pennylane(jw_active)
singles, doubles = qchem.excitations(n_elec_active, n_q_active)
hf_state = qchem.hf_state(n_elec_active, n_q_active)
n_params = len(singles) + len(doubles)

print(f'Hamiltonian: {len(H_pl.coeffs)} Pauli terms')
print(f'HF state: {hf_state}')
print(f'Singles: {len(singles)} | Doubles: {len(doubles)} | Total params: {n_params}')

In [ ]:
# ── CELL 7: VQE with Adam optimizer ──────────────────────────────────
dev = qml.device('default.qubit', wires=n_q_active)

@qml.qnode(dev, diff_method='best')
def vqe_circuit(params):
    qml.BasisState(hf_state, wires=range(n_q_active))
    qml.AllSinglesDoubles(params, wires=range(n_q_active),
                          hf_state=hf_state, singles=singles, doubles=doubles)
    return qml.expval(H_pl)

np.random.seed(0)
params = np.random.uniform(-0.01, 0.01, n_params, requires_grad=True)
opt = qml.AdamOptimizer(stepsize=0.05, beta1=0.9, beta2=0.999)

MAX_ITER = 300
CONV_TOL = 1e-9
energies = []

pbar = trange(MAX_ITER, desc='VQE (acetylene)')
for step in pbar:
    params, energy = opt.step_and_cost(vqe_circuit, params)
    energies.append(float(energy))
    pbar.set_postfix({'E': f'{energy:.8f}',
                      'ΔE_FCI': f'{abs(energy-acc.fci_energy)*1e3:.4f} mHa'})
    if step > 10 and abs(energies[-1] - energies[-2]) < CONV_TOL:
        print(f'Converged at step {step}')
        break

print(f'\nVQE  Energy : {energies[-1]:.8f} Ha')
print(f'FCI  Energy : {acc.fci_energy:.8f} Ha')
print(f'|VQE-FCI|   : {abs(energies[-1]-acc.fci_energy)*1000:.4f} mHa')
print(f'Chemical accuracy: {"✓ ACHIEVED" if abs(energies[-1]-acc.fci_energy)*1000 < 1.6 else "✗ not yet"}')

In [ ]:
# ── CELL 8: Convergence plots ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(energies, color='darkorange', lw=2.0, label='VQE (Adam, JW)')
ax.axhline(acc.fci_energy,  color='crimson',  ls='--', lw=1.5, label=f'FCI  = {acc.fci_energy:.6f} Ha')
ax.axhline(acc.ccsd_energy, color='seagreen', ls='-.', lw=1.5, label=f'CCSD = {acc.ccsd_energy:.6f} Ha')
ax.axhline(acc.hf_energy,   color='slategray',ls=':',  lw=1.5, label=f'HF   = {acc.hf_energy:.6f} Ha')
ax.set_xlabel('Optimizer Iteration', fontsize=12)
ax.set_ylabel('Energy (Ha)', fontsize=12)
ax.set_title('VQE Convergence — Acetylene (STO-3G)', fontsize=13)
ax.legend(fontsize=9)

ax2 = axes[1]
errors_mHa = [abs(e - acc.fci_energy)*1000 for e in energies]
ax2.semilogy(errors_mHa, color='darkorange', lw=2.0)
ax2.axhline(1.6, color='crimson', ls='--', lw=1.5, label='Chemical accuracy (1.6 mHa)')
ax2.set_xlabel('Optimizer Iteration', fontsize=12)
ax2.set_ylabel('|E_VQE − E_FCI| (mHa)', fontsize=12)
ax2.set_title('Error vs FCI (log scale)', fontsize=13)
ax2.legend(fontsize=9)

plt.suptitle('Acetylene · UCCSD-VQE · STO-3G · Active Space', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('acetylene_vqe_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 9: Alkyne homologous series — qubit resource table ───────────
alkyne_series = [
    {'name': 'Acetylene (C₂H₂)',  'n_e': 10, 'n_o': 5,  'pi_bonds': 2},
    {'name': 'Propyne (C₃H₄)',    'n_e': 22, 'n_o': 11, 'pi_bonds': 2},
    {'name': '1-Butyne (C₄H₆)',   'n_e': 28, 'n_o': 13, 'pi_bonds': 2},
    {'name': '1-Pentyne (C₅H₈)',  'n_e': 36, 'n_o': 16, 'pi_bonds': 2},
    {'name': '1,3-Butadiyne',     'n_e': 26, 'n_o': 12, 'pi_bonds': 4},
]

import pandas as pd
rows = []
for m in alkyne_series:
    jw_q = m['n_o'] * 2
    as_q = 4 * m['pi_bonds']  # 2 × pi_bonds orbitals each side of HOMO
    rows.append({'Molecule': m['name'], 'Electrons': m['n_e'],
                 'Orbitals': m['n_o'], 'JW Qubits': jw_q,
                 'Active-Space Q': as_q, 'π bonds': m['pi_bonds']})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ── CELL 10: Alkene vs Alkyne electron correlation comparison ─────────
# Key scientific result: triple bond has stronger multireference character

# Load ethylene reference from notebook 01 or recompute
from openfermion.chem import MolecularData
from openfermionpyscf import run_pyscf

eth_geo = [
    ('C',(0.000, 0.000, 0.000)),('C',(0.000,0.000,1.339)),
    ('H',(0.000,0.926,-0.546)),('H',(0.000,-0.926,-0.546)),
    ('H',(0.000,0.926,1.885)), ('H',(0.000,-0.926,1.885)),
]
eth = MolecularData(geometry=eth_geo, basis='sto-3g',
                    multiplicity=1, charge=0, description='ethylene')
eth = run_pyscf(eth, run_scf=True, run_ccsd=True, run_fci=True)

corr_eth = (eth.fci_energy - eth.hf_energy)*1000
corr_acc = (acc.fci_energy - acc.hf_energy)*1000

compare = pd.DataFrame({
    'Molecule':    ['Ethylene (C=C)', 'Acetylene (C≡C)'],
    'Bond order':  [2, 3],
    'π bonds':     [1, 2],
    'HF (Ha)':     [eth.hf_energy, acc.hf_energy],
    'FCI (Ha)':    [eth.fci_energy, acc.fci_energy],
    'Corr. E (mHa)': [corr_eth, corr_acc],
    'n_qubits (JW)': [eth.n_qubits, acc.n_qubits],
})
print('Alkene vs Alkyne — Electron Correlation Summary')
print(compare.to_string(index=False))
compare.to_csv('alkene_vs_alkyne.csv', index=False)
print('\nKey insight: Acetylene has stronger π-correlation → deeper VQE circuits needed.')
print(f'  Ethylene correlation energy : {corr_eth:.2f} mHa')
print(f'  Acetylene correlation energy: {corr_acc:.2f} mHa')

## Scientific Summary

| Property | Alkene (C=C) | Alkyne (C≡C) |
|---|---|---|
| Bond order | 2 | 3 |
| π electrons | 2 | 4 (two perpendicular π bonds) |
| Electron correlation | Moderate | **Stronger** |
| Circuit depth (VQE) | Shallower | Deeper |
| HOMO-LUMO gap | Larger | Smaller |

**Implication for VQE:** Acetylene requires a deeper/more expressive ansatz to
reach chemical accuracy, making it a better test case for ADAPT-VQE (Notebook 06).